# Hurricane Melissa Track Analysis (NOAA)

This notebook reads your NOAA Hurricane Melissa track files and visualizes the storm track relative to Jamaica.

It includes:
- File discovery and metadata checks
- Full track map with Jamaica context
- Jamaica-focused map (track points/line and wind swath)
- Closest-approach metrics to Jamaica


In [ ]:
from pathlib import Path
import sys

import geopandas as gpd
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import pandas as pd
from shapely.ops import unary_union


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start] + list(start.parents):
        if (p / 'dphil_papers').exists():
            return p
    raise FileNotFoundError(f'Could not find project root from {start}')


ROOT = find_project_root(Path.cwd())
robyn_libraries_path = ROOT / 'dphil_papers/robyns_libraries'
if str(robyn_libraries_path) not in sys.path:
    sys.path.append(str(robyn_libraries_path))

import Robyn_paper_2_defs

plt.style.use('default')
pd.set_option('display.max_columns', 120)


In [ ]:
track_root = ROOT / 'dphil_papers/dphil_paper_3/inputs/hurricane_melissa_track_noaa'
best_track_dir = track_root / 'al132025_best_track'

line_path = best_track_dir / 'AL132025_lin.shp'
pts_path = best_track_dir / 'AL132025_pts.shp'
windswath_path = best_track_dir / 'AL132025_windswath.shp'

jamaica_boundary_path = ROOT / 'dphil_papers/dphil_common_cross_cutting/common_incoming_data/boundaries/jamaica.gpkg'

output_dir = ROOT / 'dphil_papers/dphil_paper_3/results/hurricane_melissa_track_analysis'
output_dir.mkdir(parents=True, exist_ok=True)
SAVE_OUTPUTS = False

print('Working dir:', Path.cwd())
print('Project root:', ROOT)
print('Track root exists:', track_root.exists())
for p in [line_path, pts_path, windswath_path, jamaica_boundary_path]:
    print(p.name, 'exists ->', p.exists())
print('Output dir:', output_dir)


In [ ]:
# Discover files in the NOAA track directory
track_files = sorted([p for p in best_track_dir.glob('*') if p.is_file()])
print('Files found in best_track directory:', len(track_files))
for p in track_files:
    print('-', p.name)


In [ ]:
# Read track layers and Jamaica boundary
track_line = gpd.read_file(line_path)
track_pts = gpd.read_file(pts_path)
track_windswath = gpd.read_file(windswath_path)
jamaica = gpd.read_file(jamaica_boundary_path)

# Keep plotting in lon/lat for intuitive map axes
track_line_ll = track_line.to_crs(4326)
track_pts_ll = track_pts.to_crs(4326)
track_windswath_ll = track_windswath.to_crs(4326)
jamaica_ll = jamaica.to_crs(4326)

# Also keep projected versions (EPSG:3448) for distance calculations
track_line_3448 = track_line.to_crs(3448)
track_pts_3448 = track_pts.to_crs(3448)
jamaica_3448 = jamaica.to_crs(3448)

print('Track line rows:', len(track_line_ll), 'CRS:', track_line_ll.crs)
print('Track points rows:', len(track_pts_ll), 'CRS:', track_pts_ll.crs)
print('Track windswath rows:', len(track_windswath_ll), 'CRS:', track_windswath_ll.crs)
print('Jamaica rows:', len(jamaica_ll), 'CRS:', jamaica_ll.crs)


In [ ]:
# Parse point datetime fields for easier interpretation
pts = track_pts_ll.copy()

for c in ['YEAR', 'MONTH', 'DAY', 'HHMM', 'INTENSITY', 'MSLP']:
    if c in pts.columns:
        pts[c] = pd.to_numeric(pts[c], errors='coerce')

if all(c in pts.columns for c in ['YEAR', 'MONTH', 'DAY', 'HHMM']):
    hh = (pts['HHMM'] // 100).fillna(0).astype('Int64')
    mm = (pts['HHMM'] % 100).fillna(0).astype('Int64')

    pts['timestamp'] = pd.to_datetime(
        {
            'year': pts['YEAR'].astype('Int64'),
            'month': pts['MONTH'].astype('Int64'),
            'day': pts['DAY'].astype('Int64'),
            'hour': hh,
            'minute': mm,
        },
        errors='coerce',
        utc=True,
    )
else:
    pts['timestamp'] = pd.NaT

display_cols = [c for c in ['STORMNAME', 'STORMTYPE', 'timestamp', 'INTENSITY', 'MSLP', 'LAT', 'LON'] if c in pts.columns]
pts[display_cols].head(10)


In [ ]:
# Full track map with Jamaica context
fig, ax = plt.subplots(figsize=(11, 8), constrained_layout=True)

# Wind swath underneath for context
if len(track_windswath_ll) > 0:
    track_windswath_ll.plot(ax=ax, color='#9ecae1', alpha=0.35, edgecolor='none')

# Track line and points
track_line_ll.plot(ax=ax, color='#08519c', linewidth=2.2)
pts.plot(ax=ax, color='#08306b', markersize=18, alpha=0.8)

# Jamaica boundary
jamaica_ll.boundary.plot(ax=ax, color='black', linewidth=1.0)

ax.set_title('Hurricane Melissa NOAA Best Track with Jamaica')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')

legend_handles = [
    mpatches.Patch(facecolor='#9ecae1', edgecolor='none', alpha=0.35, label='Wind swath'),
    mpatches.Patch(facecolor='#08519c', edgecolor='none', alpha=1.0, label='Track line'),
    mpatches.Patch(facecolor='#08306b', edgecolor='none', alpha=0.8, label='Track points'),
    mpatches.Patch(facecolor='none', edgecolor='black', linewidth=1.0, label='Jamaica boundary'),
]
ax.legend(handles=legend_handles, loc='upper right', frameon=True, framealpha=0.95)

if SAVE_OUTPUTS:
    out_png = output_dir / 'hurricane_melissa_track_full_with_jamaica.png'
    fig.savefig(out_png, dpi=300)
    print('Saved:', out_png)
else:
    print('PNG export skipped (SAVE_OUTPUTS=False)')

plt.show()


In [ ]:
# Jamaica-focused map (zoom)
minx, miny, maxx, maxy = jamaica_ll.total_bounds
lon_pad = 1.5
lat_pad = 1.2

fig, ax = plt.subplots(figsize=(10, 8), constrained_layout=True)

if len(track_windswath_ll) > 0:
    track_windswath_ll.plot(ax=ax, color='#9ecae1', alpha=0.35, edgecolor='none')

track_line_ll.plot(ax=ax, color='#08519c', linewidth=2.2)
pts.plot(ax=ax, color='#08306b', markersize=22, alpha=0.85)
jamaica_ll.boundary.plot(ax=ax, color='black', linewidth=1.2)

ax.set_xlim(minx - lon_pad, maxx + lon_pad)
ax.set_ylim(miny - lat_pad, maxy + lat_pad)
ax.set_title('Hurricane Melissa Track Near Jamaica (Zoom)')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')

legend_handles = [
    mpatches.Patch(facecolor='#9ecae1', edgecolor='none', alpha=0.35, label='Wind swath'),
    mpatches.Patch(facecolor='#08519c', edgecolor='none', alpha=1.0, label='Track line'),
    mpatches.Patch(facecolor='#08306b', edgecolor='none', alpha=0.85, label='Track points'),
    mpatches.Patch(facecolor='none', edgecolor='black', linewidth=1.2, label='Jamaica boundary'),
]
ax.legend(handles=legend_handles, loc='lower left', frameon=True, framealpha=0.95)
Robyn_paper_2_defs.draw_scale_bar(
    ax,
    location=(0.88, 0.78),
    length_km=20,
    linewidth=0.6,
    label_offset=0.02,
    km_offset=0.01,
)
Robyn_paper_2_defs.draw_north_arrow(
    ax,
    location=(0.88, 0.86),
    size=0.05,
    fontsize=8,
    label_offset=0.02,
)

if SAVE_OUTPUTS:
    zoom_png = output_dir / 'hurricane_melissa_track_jamaica_zoom.png'
    fig.savefig(zoom_png, dpi=300)
    print('Saved:', zoom_png)
else:
    print('PNG export skipped (SAVE_OUTPUTS=False)')

plt.show()


In [ ]:
# Jamaica-focused map (zoom with wind speed bands)
minx, miny, maxx, maxy = jamaica_ll.total_bounds
lon_pad = 1.5
lat_pad = 1.2
wind_speed_band_styles = {
    34: {'facecolor': '#bdd7e7', 'edgecolor': '#3182bd', 'alpha': 0.45},
    50: {'facecolor': '#fdae6b', 'edgecolor': '#e6550d', 'alpha': 0.45},
    64: {'facecolor': '#fb6a4a', 'edgecolor': '#cb181d', 'alpha': 0.5},
}
wind_speed_swath_bands = track_windswath_ll.copy()
wind_speed_swath_bands['RADII'] = wind_speed_swath_bands['RADII'].astype(int)
wind_speed_swath_bands = wind_speed_swath_bands.dissolve(by='RADII', as_index=False)

fig, ax = plt.subplots(figsize=(10, 8), constrained_layout=True)

for wind_speed_knots in sorted(wind_speed_band_styles):
    wind_speed_swath = wind_speed_swath_bands[wind_speed_swath_bands['RADII'].eq(wind_speed_knots)]
    if len(wind_speed_swath) > 0:
        style = wind_speed_band_styles[wind_speed_knots]
        wind_speed_swath.plot(
            ax=ax,
            color=style['facecolor'],
            edgecolor=style['edgecolor'],
            alpha=style['alpha'],
            linewidth=0.9,
            zorder=1,
        )

track_line_ll.plot(ax=ax, color='#08519c', linewidth=2.2, zorder=3)
pts.plot(ax=ax, color='#08306b', markersize=22, alpha=0.85, zorder=4)
jamaica_ll.boundary.plot(ax=ax, color='black', linewidth=1.2, zorder=5)

ax.set_xlim(minx - lon_pad, maxx + lon_pad)
ax.set_ylim(miny - lat_pad, maxy + lat_pad)
ax.set_title('Hurricane Melissa Track Near Jamaica with Wind Speed Bands (Zoom)')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')

legend_handles = [
    mpatches.Patch(
        facecolor=wind_speed_band_styles[34]['facecolor'],
        edgecolor=wind_speed_band_styles[34]['edgecolor'],
        alpha=wind_speed_band_styles[34]['alpha'],
        label='34 kt wind-speed band',
    ),
    mpatches.Patch(
        facecolor=wind_speed_band_styles[50]['facecolor'],
        edgecolor=wind_speed_band_styles[50]['edgecolor'],
        alpha=wind_speed_band_styles[50]['alpha'],
        label='50 kt wind-speed band',
    ),
    mpatches.Patch(
        facecolor=wind_speed_band_styles[64]['facecolor'],
        edgecolor=wind_speed_band_styles[64]['edgecolor'],
        alpha=wind_speed_band_styles[64]['alpha'],
        label='64 kt wind-speed band',
    ),
    mpatches.Patch(facecolor='#08519c', edgecolor='none', alpha=1.0, label='Track line'),
    mpatches.Patch(facecolor='#08306b', edgecolor='none', alpha=0.85, label='Track points'),
    mpatches.Patch(facecolor='none', edgecolor='black', linewidth=1.2, label='Jamaica boundary'),
]
ax.legend(handles=legend_handles, loc='lower left', frameon=True, framealpha=0.95)
Robyn_paper_2_defs.draw_scale_bar(
    ax,
    location=(0.88, 0.78),
    length_km=20,
    linewidth=0.6,
    label_offset=0.02,
    km_offset=0.01,
)
Robyn_paper_2_defs.draw_north_arrow(
    ax,
    location=(0.88, 0.86),
    size=0.05,
    fontsize=8,
    label_offset=0.02,
)

if SAVE_OUTPUTS:
    wind_speed_zoom_png = output_dir / 'hurricane_melissa_track_jamaica_zoom_wind_speed_bands.png'
    fig.savefig(wind_speed_zoom_png, dpi=300)
    print('Saved:', wind_speed_zoom_png)
else:
    print('PNG export skipped (SAVE_OUTPUTS=False)')

plt.show()


In [ ]:
# Proximity summary: how close the track gets to Jamaica
jamaica_union = unary_union(jamaica_3448.geometry.tolist())
line_union = unary_union(track_line_3448.geometry.tolist())

min_distance_m = line_union.distance(jamaica_union)
min_distance_km = min_distance_m / 1000

# Distance from each track point to Jamaica
pts_3448 = track_pts_3448.copy()
pts_3448['distance_to_jamaica_km'] = pts_3448.geometry.distance(jamaica_union) / 1000

# Attach timestamp/intensity from lon/lat point table
if 'timestamp' in pts.columns:
    pts_3448['timestamp'] = pts['timestamp'].values
if 'INTENSITY' in pts.columns:
    pts_3448['INTENSITY'] = pts['INTENSITY'].values
if 'STORMTYPE' in pts.columns:
    pts_3448['STORMTYPE'] = pts['STORMTYPE'].values

closest_idx = pts_3448['distance_to_jamaica_km'].idxmin()
closest_row = pts_3448.loc[closest_idx]

summary = pd.DataFrame([
    {
        'min_track_line_distance_to_jamaica_km': round(min_distance_km, 3),
        'closest_point_distance_km': round(float(closest_row['distance_to_jamaica_km']), 3),
        'closest_point_timestamp_utc': str(closest_row.get('timestamp', pd.NaT)),
        'closest_point_intensity': closest_row.get('INTENSITY', None),
        'closest_point_stormtype': closest_row.get('STORMTYPE', None),
    }
])

summary


In [ ]:
# Optional exports
if SAVE_OUTPUTS:
    summary_csv = output_dir / 'hurricane_melissa_jamaica_proximity_summary.csv'
    points_csv = output_dir / 'hurricane_melissa_track_points_with_distance_to_jamaica.csv'

    summary.to_csv(summary_csv, index=False)
    cols = [c for c in ['timestamp', 'distance_to_jamaica_km', 'INTENSITY', 'STORMTYPE', 'geometry'] if c in pts_3448.columns]
    pts_3448[cols].to_csv(points_csv, index=False)

    print('Saved:', summary_csv)
    print('Saved:', points_csv)
else:
    print('CSV export skipped (SAVE_OUTPUTS=False)')


## Notes
- NOAA files use a geographic CRS in degrees; plotting is done in EPSG:4326.
- Distance calculations are done in EPSG:3448 (meters), then converted to km.
- Set `SAVE_OUTPUTS = True` in the config cell if you want PNG/CSV outputs written.
